# 06d · DOSE at ADM1: fixed-effects associations

Fork `fork/corridors-dose`, Part 2. Rules: [`docs/successors/dose-prespec.md`](../../docs/successors/dose-prespec.md)
§3 ("always reported"); outcomes in §4. All six first stages failed (`06b`, `06c`), so this is
descriptive only: Story 1 at regional scale.

Region-year means of night LST (day LST second) on ln(1 + mean lights) (step B) and on
ln(GRP per capita) (step C), with precipitation `P_a`, region + country × year FE, weighted by
the region's panel cells, clustered by region (country as robustness). Same rows as the first
stages.

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT))
warnings.filterwarnings("ignore")
from src.analysis import dose as D

s = D.add_lst(D.step_sample(D.build_region_panel()))
print(f"{len(s):,} region-years, {s.GID_1.nunique():,} regions, {s.GID_0.nunique()} countries, "
      f"{s.year.min()}-{s.year.max()}")

## Main: K per log point, region clusters

In [ ]:
X = {"B: lights": "log_ntl_r", "C: GRP pc": "log_grp"}
Y = {"night": "lst_night_r", "day": "lst_day_r"}
main = pd.DataFrame([dict(step=k, lst=l, **D.association(s, y, x)) for k, x in X.items() for l, y in Y.items()])
main.drop(columns=["y", "x"]).round(4)

## Robustness: country clusters

In [ ]:
cty = pd.DataFrame([dict(step=k, lst=l, **D.association(s, y, x, vcov=D.CL_CTY))
                    for k, x in X.items() for l, y in Y.items()])
cty.drop(columns=["y", "x"]).round(4)

## Information: how much the two growth measures share

Within the same FE and weights: ln lights on ln GRP. A small slope means B and C measure
different variation, so their LST associations need not agree.

In [ ]:
pd.DataFrame([D.association(s, "log_ntl_r", "log_grp")]).round(4)